# Preprocessing: Ford Used Cars

This notebook prepares the Ford used car dataset for classical machine learning regression. It cleans the raw data, creates a simple age feature, encodes categorical variables, splits the data, applies numeric scaling, and saves a cleaned dataset.

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA_PATH = Path("../data/raw/ford.csv")
PROCESSED_PATH = Path("../data/processed/ford_cleaned.csv")

## Load Dataset

In [ ]:
df = pd.read_csv(DATA_PATH)
df.head()

## Remove Duplicate Rows

In [ ]:
print("Shape before removing duplicates:", df.shape)
df_cleaned = df.drop_duplicates().copy()
print("Shape after removing duplicates:", df_cleaned.shape)

## Handle Missing Values Safely

In [ ]:
numeric_columns = df_cleaned.select_dtypes(include="number").columns
categorical_columns = df_cleaned.select_dtypes(include="object").columns

for column in numeric_columns:
    df_cleaned[column] = df_cleaned[column].fillna(df_cleaned[column].median())

for column in categorical_columns:
    mode_values = df_cleaned[column].mode()
    fill_value = mode_values.iloc[0] if not mode_values.empty else "Unknown"
    df_cleaned[column] = df_cleaned[column].fillna(fill_value)

df_cleaned.isna().sum()

Numeric missing values are filled with the median because it is robust to outliers. Categorical missing values are filled with the most common value, with `Unknown` as a fallback if a column has no mode.

## Add Feature: Car Age

In [ ]:
df_cleaned["car_age"] = 2026 - df_cleaned["year"]
df_cleaned[["year", "car_age"]].head()

`car_age` is used because vehicle age is more directly interpretable than raw registration year for price prediction. The original `year` column is kept in the cleaned CSV for clarity and traceability, but the feature matrix below uses `car_age` to avoid giving the model two almost identical time-based predictors.

## Save Cleaned Dataset

In [ ]:
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
df_cleaned.to_csv(PROCESSED_PATH, index=False)
print(f"Cleaned dataset saved to {PROCESSED_PATH}")

## Define Features and Target

In [ ]:
y = df_cleaned["price"]
X = df_cleaned.drop(columns=["price", "year"])

categorical_features = ["model", "transmission", "fuelType"]
numeric_features = [column for column in X.columns if column not in categorical_features]

print("Feature columns:", X.columns.tolist())
print("Target column: price")
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

## Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

## Encoding and Scaling Setup

In [ ]:
try:
    categorical_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    categorical_encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_features),
        ("categorical", categorical_encoder, categorical_features),
    ]
)

preprocessing_pipeline = Pipeline(
    steps=[("preprocessor", preprocessor)]
)

X_train_prepared = preprocessing_pipeline.fit_transform(X_train)
X_test_prepared = preprocessing_pipeline.transform(X_test)

print("Prepared training feature matrix shape:", X_train_prepared.shape)
print("Prepared test feature matrix shape:", X_test_prepared.shape)

`StandardScaler` is applied to numeric columns because many classical regression models are sensitive to feature scale. Categorical columns are encoded using one-hot encoding so models can use `model`, `transmission`, and `fuelType` without treating them as ordered numbers.